In [ ]:
import json
import base64
import io
import socket
import asyncio
import threading
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
from PIL import Image
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

nest_asyncio.apply()

# Helper function to dynamically find an open, clean port
def find_free_port():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(('127.0.0.1', 0))
    port = s.getsockname()[1]
    s.close()
    return port

# 2. Authenticate tunnel using your token link
NGROK_AUTH_TOKEN = "3Jfmp3Pzg6S64P9zCIpftebcv6B_4f2QXrAMoKL5LGtBjDvMS"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("Loading Open-Source Vision-Language Model onto T4 GPU...")
model_id = "Qwen/Qwen2-VL-2B-Instruct" 
model = Qwen2VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
processor = AutoProcessor.from_pretrained(model_id)

app = FastAPI()

class Payload(BaseModel):
    image: str
    prompt: str

@app.post("/analyze_colab")
async def analyze_colab(data: Payload):
    try:
        # Securely decode incoming base64 image streams
        img_bytes = base64.b64decode(data.image)
        image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": data.prompt}
                ]
            }
        ]
        
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to("cuda")
        
        # FIXED GENERATION PARAMS: Explicitly sampling to read text line-by-line cleanly
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=1500,
            do_sample=True,
            temperature=0.1,  # Forces exact character transcription focus
            top_p=0.9
        )
        
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        
        # CRITICAL DECODING FIX: Changed from batch_decode to single token array decode
        output_text = processor.decode(generated_ids_trimmed[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
        
        # Locate absolute JSON dictionary layout string boundaries
        json_start = output_text.find('{')
        json_end = output_text.rfind('}') + 1
        
        if json_start == -1 or json_end == 0:
            raise Exception(f"Raw response text failed structural checks: {output_text}")
            
        json_data = json.loads(output_text[json_start:json_end])
        return {"status": "success", "data": json_data}
        
    except Exception as e:
        print(f"❌ CRITICAL EXTRACTION FAILURE: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

# 3. Terminate old channels and initialize a fresh dynamic port tunnel link
ngrok.kill() 
free_port = find_free_port()
public_url = ngrok.connect(free_port)
print(f"\n🚀 SUCCESS! YOUR PUBLIC BRIDGE TUNNEL IS LIVE:")
print(f"👉 {public_url.public_url} 👈\n")

# 4. Spin up the server loop on an independent background thread using the clean port
config = uvicorn.Config(app, host="127.0.0.1", port=free_port, loop="asyncio")
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run)
thread.start()
print(f"🎉 Cloud server is actively listening on fresh port {free_port} background thread!")